In [1]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


In [2]:
# Make sure that the working directory is the project root.
%cd -q ../

In [3]:
import torch
import pickle
import numpy as np
from autoregltl import ted, llama, mamba, dataset
from autoregltl.ltl import trace_check

from tqdm.auto import tqdm
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec

mpl.rcParams['figure.dpi'] = 192
mpl.rcParams['svg.fonttype'] = 'none'  # Critical: don't convert text to paths
# plt.rcParams['pdf.use14corefonts'] = True

device = torch.device('cuda')

In [4]:
redgreen = mpl.colors.LinearSegmentedColormap(
    "redgreen",
    {
        'red': (
            (0.0, 1.0, 1.0),
            (0.15873*3, 1.0, 1.0),
            (0.174603*3, 0.96875, 0.96875),
            (1.0, 0.0, 0.0),
        ),
        'green': (
            (0.0, 0.0, 0.0),
            (0.15873*3, 0.9375, 0.9375),
            (0.174603*3, 1.0, 1.0),
            (1.0, 1.0, 1.0),
        ),
        'blue': (
            (0.0, 0.0, 0.0),
            (1.0, 0.0, 0.0),
        ),
    }
)

In [5]:
def read_eval2da(filename):
    with open(filename, 'rb') as f: evaldict = pickle.load(f)
    #torch.load(filename, map_location=torch.device('cpu'))
    # dict_keys(['correct_matrix', 'count_matrix', 'correct', 'count', 'repeat_count', 'eval_ds', 'all_results'])
        
    count_matrix = evaldict['count_matrix']
    correct_matrix = evaldict['correct_matrix']
    sample_rate = count_matrix / 100.0
    eval_results = torch.where(count_matrix > 0, correct_matrix / count_matrix, 0.0)

    return eval_results, sample_rate, evaldict['correct'], evaldict['count']

In [6]:
import json, os, re

def extract_model_info(eval_file_path):
    """Extract model display name (with blanks), active components set, and number of parameters from config.json and command-log.txt"""
    model_dir = os.path.dirname(eval_file_path)
    config_path = os.path.join(model_dir, 'config.json')
    command_log_path = os.path.join(model_dir, 'command-log.txt')
    
    # Parse number of parameters from command-log.txt
    num_params = None
    try:
        with open(command_log_path, 'r') as f:
            first_line = f.readline().strip()
            # Extract number from "Number of parameters: 2_788_288"
            match = re.search(r'Number of parameters: ([\d_]+)', first_line)
            if match:
                num_params = int(match.group(1).replace('_', ''))
    except Exception as e:
        print(f"Warning: Could not read command-log.txt at {command_log_path}: {e}")
    
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
    except Exception as e:
        print(f"Error reading config.json at {config_path}: {e}")
        return None, set()
    
    if "dynamic_aps" in config.get("vocab", {}):
        return num_params, set()
    
    active = set()

    # Check for enc_per (EP) - enabled by default unless no_enc_per is true
    if not config.get('no_enc_per', False):
        active.add('EP')
    
    # Check for dec_per (DP) - enabled by default unless no_dec_per is true
    if not config.get('no_dec_per', False):
        active.add('DP')
    
    # Check for enc_agg (EA) - enabled by default unless no_enc_agg is true
    if not config.get('no_enc_agg', False):
        active.add('EA')
    
    # Check for dec_agg (DA) - enabled by default unless no_dec_agg is true
    if not config.get('no_dec_agg', False):
        active.add('DA')
    
    # Check cross_attn for CP and CA
    cross_attn = config.get('cross_attn', '')
    if 'per' in cross_attn:
        active.add('CP')
    if 'agg' in cross_attn:
        active.add('CA')
    
    return num_params, active

def build_name_from_active(active_components, delim=' '):
    """Build display name with blanks following component order, matching extract_model_info output."""
    ordered = ['EP', 'DP', 'EA', 'DA', 'CP', 'CA']
    parts = []
    for comp in ordered:
        parts.append(comp if comp in active_components else '  ')
    return delim.join(parts)


def build_filename_from_active(active_components):
    ordered = ['EP', 'DP', 'EA', 'DA', 'CP', 'CA']
    parts = []
    for comp in ordered:
        if comp in active_components:
            parts.append(comp)
    return '-'.join(parts)

In [7]:
def find_eval2da_files(root_dir, check_prop=False):
    eval_files = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename == 'eval2da1.pkl':
                if (not check_prop) != ("-prop" in dirpath):
                    if check_prop:
                        if not "s1-" in dirpath:
                            continue
                    else:
                        if "-nocos" in dirpath:
                            continue
                    eval_files.append(os.path.join(dirpath, filename))
    return eval_files

def compute_accuracy(file_path):
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            correct = data.get('correct', 0)
            count = data.get('count', 1)  # avoid division by zero
            accuracy = (correct / count) * 100.0
            return accuracy
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def read_summary_json(model_dir, result_name='ltl-35-val10k-b3'):
    """
    Reads the 'results/ltl-35-val10k-b3/summary.json' file from the model directory.
    Returns a dictionary of results.
    """
    summary_path = os.path.join(model_dir, 'results', result_name, 'summary.json')
    if os.path.exists(summary_path):
        with open(summary_path, 'r') as f:
            return json.load(f)
    return {}

In [8]:
def plot_heatmap(model_path, active_components, parameters, rank, prop: bool):
    figscale = 0.875
    fig = plt.figure(figsize=(7.5 * figscale, 2.75 * figscale), layout="constrained")
    gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[3, 5])

    ax_table = fig.add_subplot(gs[0, 0])
    ax = fig.add_subplot(gs[0, 1])
    ax_table.axis("off")

    components = {
        ("Encoder", "Per-Stream"): "EP" in active_components,
        ("Encoder", "Aggregated"): "EA" in active_components,
        ("Decoder", "Per-Stream"): "DP" in active_components,
        ("Decoder", "Aggregated"): "DA" in active_components,
        ("Cross",   "Per-Stream"): "CP" in active_components,
        ("Cross",   "Aggregated"): "CA" in active_components,
    }

    common_kwargs = dict(
        aspect='auto',
    )
    xticks = [1, 10, 20, 30, 40, 50]
    yticks = [1, 5, 10]

    eval_results, sample_rate, correct, count = read_eval2da(model_path)
    if prop:
        eval_results = eval_results[1:, :]
        sample_rate = sample_rate[1:, :]
    ax.imshow(eval_results, cmap=redgreen, vmin=0.0, vmax=1.0, **common_kwargs)
    # Plotting the modulus array as the 'value' part
    black = torch.zeros(10+int(not prop), 50, 4)
    black[:, :, -1] = 1.0 - sample_rate
    #black[:, :, -1] = torch.where(sample_rate > 0, 0.0, 1.0)
    ax.imshow(black, **common_kwargs)

    ax.set_ylabel("AP count")
    ax.set_xlabel("Formula length")
    ax.set_xticks([i -1 for i in xticks], xticks)
    if prop:
        ax.set_yticks([i-1 for i in yticks], yticks)
    else:
        ax.set_yticks(yticks, yticks)
    # # 35 is not actually inclusive
    # ax.add_patch(mpl.patches.Rectangle((-0.4, -0.4), 35-.1, 5+int(not prop)-.1, fill=False, edgecolor='white', lw=1.5, linestyle = 'dashed'))
    ax.set_title(f"Accuracy: {100.0*correct/count:.2f}%")

    fig.colorbar(plt.cm.ScalarMappable(cmap=redgreen), ax=ax, pad=0.0125)

    # TABLE

    def yn_emoji(v):
        # return "✔" if v else "✖"   # fallback: "✔" / "✖"
        return "✔" if v else " "   # fallback: "✔" / "✖"

    rows = []
    for block in ["Encoder", "Decoder", "Cross"]:
        rows.append([block, "Per-Stream", yn_emoji(components[(block, "Per-Stream")])])
        rows.append(["",     "Aggregated", yn_emoji(components[(block, "Aggregated")])])

    rows.append(["", f"Parameters: {parameters:,}", ""])
    rows.append(["", f"Rank: {rank}", ""])

    table = ax_table.table(
        cellText=rows,
        colLabels=["Attention Type", "", "Enabled"],
        loc="center",
        cellLoc="center",
        colLoc="center",
        edges="horizontal",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.1, 1.6)
    # Header styling
    for j in range(3):
        table[(0, j)].set_text_props(weight='bold')
        
    for i in range(1, 7):
        table[(i, 2)].set_text_props(weight='bold')
        table[(i, 2)].get_text().set_color("red" if table[(i, 2)].get_text().get_text() == "✖" else "#00AA00")
        table[(i, 2)].get_text().set_fontsize(15)

    # need to draw here so the text positions are calculated
    fig.canvas.draw()

    top = table[(0, 0)]
    bottom = table[(0, 1)]
    x, y = top.get_text().get_position()
    x2, y2 = bottom.get_text().get_position()
    top.get_text().set_transform(mpl.transforms.Affine2D().translate((x2-x)/ 5, 0))

    # Merge visual blocks (rows 1–2, 3–4, 5–6 in table indexing)
    merge_starts = [1, 3, 5]

    for r in merge_starts:
        top = table[(r, 0)]
        bottom = table[(r+1, 0)]

        bottom.get_text().set_text("")      # remove duplicate label
        top.visible_edges = "T"           # hide bottom edge
        bottom.visible_edges = "B"        # hide top edge
        # --- vertical centering trick ---
        #top.get_text().set_va("bottom")   # push text downward
        x, y = top.get_text().get_position()
        x2, y2 = bottom.get_text().get_position()
        top.get_text().set_transform(mpl.transforms.Affine2D().translate(0, (y2-y)/ 5))

    for i in range(7, 9):
        for j in range(3):
            table[(i, j)].visible_edges = ""
            table[(i, j)].get_text().set_transform(mpl.transforms.Affine2D().translate(0, -21/3))

    output_dir = "figures/ablation/" + ("prop" if prop else "ltl")
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, build_filename_from_active(active_components) + ".pdf"))
    plt.close()

In [19]:
import itertools
from collections import defaultdict

def generate_figures(prop: bool):
	is_prop = prop
	root_dir = 'models-scc'
	files = find_eval2da_files(root_dir, check_prop=is_prop)
	results = defaultdict(list)

	for file_path in files:
		accuracy = compute_accuracy(file_path)
		if accuracy is not None:
			parameters, active_components = extract_model_info(file_path)
			if len(active_components) == 0:
				continue  # Skip models with no active components
			model_name = build_name_from_active(active_components)
			results[model_name].append((
				file_path,
				accuracy,
				parameters,
				active_components,
				read_summary_json(os.path.dirname(file_path)).get('correct', 0) / 100.0,
				read_summary_json(os.path.dirname(file_path), "ltl-35-10ap-val10k-b3").get('correct', 0) / 100.0,
			))

	# Sort each value of results by accuracy descending
	for model_name in results:
		results[model_name].sort(key=lambda x: x[1], reverse=True)

	# Find the best model_name across combinations
	best_accuracy = max(
		(entries[0][1] for entries in results.values()),
		default=None
	)
	best_name = None
	for model_name, entries in results.items():
		if entries[0][1] == best_accuracy:
			best_name = model_name
			break

	# Coverage check for component combinations
	group2_options = [{'EP'}, {'EA'}, {'EP', 'EA'}]
	group3_options = [{'DP'}, {'DA'}, {'DP', 'DA'}]
	group4_options = [{'CP'}, {'CP', 'CA'}]
	# group4_options = [{'CP'}, {'CA'}, {'CP', 'CA'}]

	valid_combos = [{'EP', 'EA', 'DP', 'DA', 'CA'}]
	for g2, g3, g4 in itertools.product(group2_options, group3_options, group4_options):
		active = set().union(g2, g3, g4)
		valid_combos.append(active)

	valid_names = sorted([build_name_from_active(active) for active in valid_combos], reverse=True, key=lambda name: results.get(name, [(None, -1)])[0][1])

	best_entries = []
	for i, model_name in enumerate(valid_names):
		if model_name in results:
			best_entry = results[model_name][0]
			best_entries.append(best_entry)
			# print(f"{model_name}: {best_entry[1]:.2f}%, {best_entry[4]:.2f}%, {best_entry[5]:.2f}%, Params: {best_entry[2]}, Path: {best_entry[0]}")
			# plot_heatmap(
			# 	best_entry[0],
			# 	best_entry[3],
			# 	best_entry[2],
			# 	i+1,
			# 	prop=is_prop
			# )
		else:
			print(f"{model_name}: No results found.")
			best_entries.append(None)
	print()
	
	for best_entry in best_entries:
		if best_entry is None:
			print("")
			continue
		comps = build_name_from_active(best_entry[3], delim=' & ')
		refname = ("prop-" if is_prop else "ltl-") + build_filename_from_active(best_entry[3])
		print(f"& {comps} & {best_entry[4]:.2f}\% & {best_entry[5]:.2f}\% & {best_entry[1]:.2f}\% & {best_entry[2]:,} & \\ref{{fig:{refname}}} \\\\ % Path: {best_entry[0]}")

In [20]:
generate_figures(prop=False)


& EP & DP & EA &    & CP &    & 98.12\% & 96.53\% & 90.47\% & 2,654,144 & \ref{fig:ltl-EP-DP-EA-CP} \\ % Path: models-scc/reverse-epat/ns0-per-noda-s46/eval2da1.pkl
& EP & DP & EA &    & CP & CA & 98.23\% & 96.49\% & 90.27\% & 2,788,288 & \ref{fig:ltl-EP-DP-EA-CP-CA} \\ % Path: models-scc/reverse-epat/ns0-peragg-noda-s42/eval2da1.pkl
& EP & DP &    & DA & CP &    & 97.96\% & 96.32\% & 89.66\% & 2,654,144 & \ref{fig:ltl-EP-DP-DA-CP} \\ % Path: models-scc/reverse-epat/ns0-per-noea-s42/eval2da1.pkl
& EP & DP & EA & DA & CP &    & 98.33\% & 96.45\% & 89.47\% & 2,788,288 & \ref{fig:ltl-EP-DP-EA-DA-CP} \\ % Path: models-scc/reverse-epat/ns0-per-s92/eval2da1.pkl
& EP & DP &    &    & CP & CA & 97.99\% & 96.05\% & 89.26\% & 2,654,144 & \ref{fig:ltl-EP-DP-CP-CA} \\ % Path: models-scc/reverse-epat/nerc/ns0-peragg-noeda-s42/eval2da1.pkl
& EP & DP & EA & DA & CP & CA & 97.96\% & 96.32\% & 89.16\% & 2,922,432 & \ref{fig:ltl-EP-DP-EA-DA-CP-CA} \\ % Path: models-scc/reverse-epat/ns0-peragg-s42/eval2

In [21]:

generate_figures(prop=True)


& EP & DP & EA & DA & CP &    & 97.94\% & 97.63\% & 95.05\% & 2,906,496 & \ref{fig:prop-EP-DP-EA-DA-CP} \\ % Path: models-scc/reverse-prop-epat/s1-43/eval2da1.pkl
& EP & DP & EA & DA & CP & CA & 96.75\% & 96.05\% & 92.66\% & 3,131,136 & \ref{fig:prop-EP-DP-EA-DA-CP-CA} \\ % Path: models-scc/reverse-prop-epat/s1-peragg-13/eval2da1.pkl
& EP & DP &    & DA & CP &    & 96.84\% & 96.19\% & 92.47\% & 2,681,856 & \ref{fig:prop-EP-DP-DA-CP} \\ % Path: models-scc/reverse-prop-epat/s1-noea-90/eval2da1.pkl
&    & DP & EA & DA & CP &    & 96.29\% & 95.16\% & 91.44\% & 2,681,856 & \ref{fig:prop-DP-EA-DA-CP} \\ % Path: models-scc/reverse-prop-epat/s1-noep-42/eval2da1.pkl
& EP & DP &    & DA & CP & CA & 95.69\% & 94.95\% & 91.10\% & 2,906,496 & \ref{fig:prop-EP-DP-DA-CP-CA} \\ % Path: models-scc/reverse-prop-epat/nerc/s1-peragg-noea-42/eval2da1.pkl
&    & DP & EA & DA & CP & CA & 94.81\% & 93.48\% & 89.15\% & 2,906,496 & \ref{fig:prop-DP-EA-DA-CP-CA} \\ % Path: models-scc/reverse-prop-epat/s1-peragg

In [12]:
group2_options = [{'EP'}, {'EA'}, {'EP', 'EA'}]
group3_options = [{'DP'}, {'DA'}, {'DP', 'DA'}]
group4_options = [{'CP'}, {'CP', 'CA'}]
# group4_options = [{'CP'}, {'CA'}, {'CP', 'CA'}]

valid_combos = [{'EP', 'EA', 'DP', 'DA', 'CA'}]
for g2, g3, g4 in itertools.product(group2_options, group3_options, group4_options):
	active = set().union(g2, g3, g4)
	valid_combos.append(active)

valid_names = sorted([
	(build_name_from_active(active), build_filename_from_active(active))
  for active in valid_combos
  ], reverse=True, key=lambda a: a[0])

for name, filename in valid_names:
	print(f"\\begin{{subfigure}}[b]{{\\textwidth}}\\includegraphics[width=\\textwidth]{{figs/ablation/prop/{filename}.pdf}}\\caption{{{filename}}}\\label{{fig:prop-{filename}}}\\end{{subfigure}}")

\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-EA-DA-CP-CA.pdf}\caption{EP-DP-EA-DA-CP-CA}\label{fig:prop-EP-DP-EA-DA-CP-CA}\end{subfigure}
\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-EA-DA-CP.pdf}\caption{EP-DP-EA-DA-CP}\label{fig:prop-EP-DP-EA-DA-CP}\end{subfigure}
\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-EA-DA-CA.pdf}\caption{EP-DP-EA-DA-CA}\label{fig:prop-EP-DP-EA-DA-CA}\end{subfigure}
\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-EA-CP-CA.pdf}\caption{EP-DP-EA-CP-CA}\label{fig:prop-EP-DP-EA-CP-CA}\end{subfigure}
\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-EA-CP.pdf}\caption{EP-DP-EA-CP}\label{fig:prop-EP-DP-EA-CP}\end{subfigure}
\begin{subfigure}[b]{\textwidth}\includegraphics[width=\textwidth]{figs/ablation/prop/EP-DP-DA-CP-CA.pdf}\caption{EP-DP-